In [ ]:
# ============================================================
# ASSIGNMENT 2 - A2_03f: HYBRID NRMS (text + behavioural features) - MIND, LARGE
# Team: Shivam Patel + Suyash Pande | CS4.406 IRE
#
# The decisive neural experiment: give NRMS EVERY advantage -
#   - its best text config (title + abstract)
#   - PLUS the behavioural features it was blind to (popularity, CTR, recency,
#     freshness, category-match, position, slate-size)
# fused: final_score = text_score(user,cand) + feature_branch(behavioural_feats).
# Features are computed HERE (aligned to NRMS samples), point-in-time (leakage-safe),
# and STANDARDISED before fusion.
#
# Purpose: if hybrid NRMS STILL loses to GBDT (0.71), we can state that the neural model
# fails even when given the same signals -> trees dominate tabular behavioural features.
#
# Same setup: 800k cap, 2 epochs, lr=1e-4 + grad clip. Read-no-token / write-token.
# Saves: mind_nrms_hybrid_ep*.pt, _best.pt, _devscores.pkl
# NEEDS: Kaggle MIND-large mounted.
# ============================================================
!pip install huggingface_hub sentence-transformers -q
import os, glob, re, numpy as np, pandas as pd, datetime as dt, random, time, pickle
from bisect import bisect_left
from collections import defaultdict, Counter
import torch, torch.nn as nn, torch.nn.functional as Fnn
from huggingface_hub import hf_hub_download, HfApi
random.seed(0); torch.manual_seed(0)

device=torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:",device)
HF_REPO="donbosoc/mind-artifacts"
WRITE_TOKEN=None
try:
    from kaggle_secrets import UserSecretsClient
    WRITE_TOKEN=UserSecretsClient().get_secret("HF_TOKEN"); print("write token loaded")
except Exception as e:
    print("no HF_TOKEN -> uploads skipped:", e)
def dl(name): return hf_hub_download(HF_REPO, name, repo_type="dataset")
def push(local,name):
    if WRITE_TOKEN is None: print("  skip upload:",name); return
    try:
        HfApi(token=WRITE_TOKEN).upload_file(path_or_fileobj=local,path_in_repo=name,repo_id=HF_REPO,repo_type="dataset"); print("  pushed",name)
    except Exception as e: print("  upload failed",name,"->",e)

LTR=glob.glob("/kaggle/input/**/MINDlarge_train",recursive=True)[0]
LDV=glob.glob("/kaggle/input/**/MINDlarge_dev",recursive=True)[0]
LR=1e-4; GRAD_CLIP=1.0; EPOCHS=2; BATCH=64; NPRATIO=4
TRAIN_CAP=800_000; MAX_LEN=50; MAX_HIST=30; EARLY_STOP_PATIENCE=1


In [ ]:
# ---- vocab (title+abstract) + article metadata for features ----
NEWS=["news_id","category","subcategory","title","abstract","url","te","ae"]
BEH =["impression_id","user_id","time","history","impressions"]
_WORD=re.compile(r"[^\W\d_]+", re.UNICODE)
def tok(t): return _WORD.findall(t.lower()) if isinstance(t,str) else []
def pfx(x): return f"mind:{x}"
news=pd.concat([pd.read_csv(f"{d}/news.tsv",sep="\t",header=None,names=NEWS,quoting=3,
               usecols=["news_id","category","title","abstract"]) for d in (LTR,LDV)]).drop_duplicates("news_id").reset_index(drop=True)
news["title"]=news["title"].fillna(""); news["abstract"]=news["abstract"].fillna(""); news["category"]=news["category"].fillna("")
text_tokens={pfx(r.news_id): tok(f"{r.title} {r.abstract}")[:MAX_LEN] for r in news.itertuples()}
vocab={"<pad>":0,"<unk>":1}
for tks in text_tokens.values():
    for w in tks:
        if w not in vocab: vocab[w]=len(vocab)
art_ids=[pfx(r.news_id) for r in news.itertuples()]; art_row={a:i for i,a in enumerate(art_ids)}
cat_lut={pfx(r.news_id):r.category for r in news.itertuples()}
def enc(aid):
    idx=[vocab.get(w,1) for w in text_tokens.get(aid,[])][:MAX_LEN]
    return idx+[0]*(MAX_LEN-len(idx))
text_idx=np.array([enc(a) for a in art_ids],dtype=np.int64)

# MiniLM embeddings for the content-similarity features
from sentence_transformers import SentenceTransformer
minilm=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb=minilm.encode([f"{r.title} {r.abstract}".strip() for r in news.itertuples()],
                  batch_size=512,normalize_embeddings=True,convert_to_numpy=True,show_progress_bar=True)
emb_by_id={art_ids[i]:emb[i] for i in range(len(art_ids))}
print("vocab:",len(vocab),"| embeddings:",emb.shape)


In [ ]:
# ---- point-in-time event index (LEAKAGE-SAFE: exclude the current impression's own events) ----
def load_beh(p):
    b=pd.read_csv(f"{p}/behaviors.tsv",sep="\t",header=None,names=BEH,quoting=3)
    b["t"]=pd.to_datetime(b["time"],format="%m/%d/%Y %I:%M:%S %p",errors="coerce"); return b
b_tr=load_beh(LTR); b_dv=load_beh(LDV)
hist_lut={}
for b in (b_tr,b_dv):
    for u,h in zip(b["user_id"],b["history"]):
        if isinstance(h,str) and h: hist_lut[pfx(u)]=[pfx(x) for x in h.split()]

# store per-article sorted lists of (time, impression_id). Counting excludes the SAME iid,
# so a candidate can never "see" its own impression's click/impression -> no label leak.
click_ev=defaultdict(list); imp_ev=defaultdict(list); first_seen={}
for b in (b_tr,b_dv):
    for iid,t,imps in zip(b["impression_id"],b["t"],b["impressions"]):
        if pd.isna(t) or not isinstance(imps,str): continue
        for tkn in imps.split():
            p=tkn.split("-"); nid=pfx(p[0])
            if nid not in first_seen or t<first_seen[nid]: first_seen[nid]=t
            if len(p)==2:
                imp_ev[nid].append((t,iid))
                if p[1]=="1": click_ev[nid].append((t,iid))
for d in (click_ev,imp_ev):
    for k in d: d[k].sort(key=lambda z: z[0])
# pre-extract sorted time arrays for bisect
click_t={k:[z[0] for z in v] for k,v in click_ev.items()}
imp_t  ={k:[z[0] for z in v] for k,v in imp_ev.items()}

def cnt(evlist, tarr, aid, T, cur_iid, w=None):
    """count events strictly before T, EXCLUDING any from cur_iid (the current impression)."""
    tl=tarr.get(aid)
    if not tl: return 0
    hi=bisect_left(tl,T)
    lo=0 if w is None else bisect_left(tl,T-dt.timedelta(hours=w))
    if hi<=lo: return 0
    # subtract events in [lo,hi) that belong to cur_iid
    evs=evlist[aid]
    same=sum(1 for i in range(lo,hi) if evs[i][1]==cur_iid)
    return (hi-lo)-same

def ctr(aid,T,cur_iid,w=None):
    c=cnt(click_ev,click_t,aid,T,cur_iid,w); s=cnt(imp_ev,imp_t,aid,T,cur_iid,w)
    return c/s if s>0 else 0.0
def freshness(aid,T,tau=6.0):
    fs=first_seen.get(aid)
    if fs is None or T is None: return 0.0
    dh=(T-fs).total_seconds()/3600.0
    return float(np.exp(-dh/tau)) if dh>=0 else 0.0
def user_cat(uid,mh=50):
    ai=hist_lut.get(uid,[])[-mh:]; cats=[cat_lut.get(a) for a in ai]
    tot=len([c for c in cats if c]); cc=Counter(c for c in cats if c)
    return {k:v/tot for k,v in cc.items()} if tot else {}
FDIM=8
def feat_vec(uid,T,iid,c,i,m,um,hv):
    cv=emb_by_id.get(c)
    em=float(um@cv) if (um is not None and cv is not None) else 0.0
    p24=cnt(click_ev,click_t,c,T,iid,24); ptot=cnt(click_ev,click_t,c,T,iid)
    c24=ctr(c,T,iid,24); ctot=ctr(c,T,iid); fr=freshness(c,T)
    cm=user_cat(uid).get(cat_lut.get(c,""),0.0); pos=i/max(1,m-1)
    return [em, np.log1p(p24), np.log1p(ptot), c24, ctot, fr, cm, pos]
print("leakage-safe feature functions ready (same-impression events excluded)")


In [ ]:
# ---- HYBRID NRMS: text score + standardised behavioural feature branch ----
class AdditiveAttention(nn.Module):
    def __init__(s,dim,hid=200):
        super().__init__(); s.p=nn.Linear(dim,hid); s.q=nn.Linear(hid,1,bias=False)
    def forward(s,x,mask=None):
        e=torch.tanh(s.p(x)); a=s.q(e).squeeze(-1)
        if mask is not None: a=a.masked_fill(mask==0,-1e9)
        return (x*torch.softmax(a,-1).unsqueeze(-1)).sum(1)
class NewsEncoder(nn.Module):
    def __init__(s,V,ed=300,h=15,hd=20):
        super().__init__(); d=h*hd
        s.emb=nn.Embedding(V,ed,padding_idx=0); s.attn=nn.MultiheadAttention(ed,h,batch_first=True)
        s.proj=nn.Linear(ed,d); s.add=AdditiveAttention(d); s.drop=nn.Dropout(0.2)
    def forward(s,t):
        m=(t!=0); x=s.drop(s.emb(t)); x,_=s.attn(x,x,x,key_padding_mask=~m)
        x=s.drop(x); x=torch.relu(s.proj(x)); return s.add(x,m)
class UserEncoder(nn.Module):
    def __init__(s,d):
        super().__init__(); s.attn=nn.MultiheadAttention(d,5,batch_first=True); s.add=AdditiveAttention(d)
    def forward(s,hv,mask):
        x,_=s.attn(hv,hv,hv,key_padding_mask=~mask); return s.add(x,mask)
class HybridNRMS(nn.Module):
    def __init__(s,V,fdim,h=15,hd=20):
        super().__init__(); s.news=NewsEncoder(V,h=h,hd=hd); s.user=UserEncoder(h*hd)
        s.fbn=nn.BatchNorm1d(fdim)                      # standardise features
        s.fbranch=nn.Sequential(nn.Linear(fdim,64),nn.ReLU(),nn.Dropout(0.2),nn.Linear(64,1))
    def forward(s,ht,hm,ct,feats):
        B,H,L=ht.shape; C=ct.shape[1]
        hv=s.news(ht.reshape(B*H,L)).reshape(B,H,-1); u=s.user(hv,hm)
        cv=s.news(ct.reshape(B*C,L)).reshape(B,C,-1)
        text_score=torch.bmm(cv,u.unsqueeze(-1)).squeeze(-1)          # (B,C)
        f=feats.reshape(B*C,-1); f=s.fbn(f); fs=s.fbranch(f).reshape(B,C)  # (B,C)
        return text_score + fs                                        # fuse
model=HybridNRMS(len(vocab),FDIM).to(device)
print("Hybrid NRMS params:", sum(p.numel() for p in model.parameters()))


In [ ]:
# ---- samples + collate that also builds the feature tensor ----
def make_samples(b_df):
    S=[]
    for iid,u,t,imps in zip(b_df["impression_id"],b_df["user_id"],b_df["t"],b_df["impressions"]):
        if not isinstance(imps,str) or pd.isna(t): continue
        uid=pfx(u); hist=hist_lut.get(uid)
        if not hist: continue
        pos=[pfx(x.split("-")[0]) for x in imps.split() if x.endswith("-1")]
        neg=[pfx(x.split("-")[0]) for x in imps.split() if x.endswith("-0")]
        if not pos or not neg: continue
        for p in pos:
            ng=random.sample(neg,NPRATIO) if len(neg)>=NPRATIO else neg+[random.choice(neg) for _ in range(NPRATIO-len(neg))]
            S.append((uid,t,iid,p,ng))
        if len(S)>=TRAIN_CAP: break
    return S
train_samples=make_samples(b_tr)
random.Random(0).shuffle(train_samples); train_samples=train_samples[:TRAIN_CAP]
print("train samples (capped):",len(train_samples))
def hist_tensor(uid):
    h=hist_lut.get(uid,[])[-MAX_HIST:]
    rows=[art_row[a] for a in h if a in art_row][:MAX_HIST]
    mask=[1]*len(rows)+[0]*(MAX_HIST-len(rows)); rows=rows+[0]*(MAX_HIST-len(rows))
    return rows,mask
def uvec(uid):
    hv=[emb_by_id[a] for a in hist_lut.get(uid,[])[-50:] if a in emb_by_id]
    um=(np.mean(hv,0)/(np.linalg.norm(np.mean(hv,0))+1e-9)) if hv else None
    return um,hv
def collate(batch):
    H=[];HM=[];C=[];F=[]
    for uid,T,iid,pos,negs in batch:
        hr,hm=hist_tensor(uid); H.append(hr);HM.append(hm)
        cand=[pos]+negs; C.append([art_row.get(c,0) for c in cand])
        um,hv=uvec(uid); m=len(cand)
        F.append([feat_vec(uid,T,iid,c,i,m,um,hv) for i,c in enumerate(cand)])
    H=np.array(H);C=np.array(C)
    return (torch.tensor(text_idx[H]),torch.tensor(np.array(HM),dtype=torch.bool),
            torch.tensor(text_idx[C]),torch.tensor(np.array(F),dtype=torch.float32),
            torch.zeros(len(batch),dtype=torch.long))
def _auc(s,lb):
    p=lb==1;n=lb==0;np_,nn=p.sum(),n.sum()
    if np_==0 or nn==0: return None
    o=np.argsort(s);r=np.empty_like(o,float);r[o]=np.arange(1,len(s)+1)
    return float((r[p].sum()-np_*(np_+1)/2)/(np_*nn))
def _mrr(s,lb):
    for rank,idx in enumerate(np.argsort(-s),1):
        if lb[idx]==1: return 1.0/rank
    return 0.0
def _ndcg(s,lb,k):
    o=np.argsort(-s)[:k];g=lb[o]
    dcg=sum(gg/np.log2(i+2) for i,gg in enumerate(g))
    idcg=sum(gg/np.log2(i+2) for i,gg in enumerate(np.sort(lb)[::-1][:k]))
    return float(dcg/idcg) if idcg>0 else 0.0
dev_impr=[]
for iid,u,t,imps in zip(b_dv["impression_id"],b_dv["user_id"],b_dv["t"],b_dv["impressions"]):
    if not isinstance(imps,str) or pd.isna(t): continue
    uid=pfx(u)
    if not hist_lut.get(uid): continue
    cand=[pfx(x.split("-")[0]) for x in imps.split()]
    labs=np.array([1 if x.endswith("-1") else 0 for x in imps.split()])
    if labs.sum()>0: dev_impr.append((iid,uid,t,cand,labs))
random.Random(0).shuffle(dev_impr); dev_impr=dev_impr[:40000]
print("dev eval set:",len(dev_impr))
@torch.no_grad()
def eval_h(cache=None):
    model.eval(); A=[];M=[];N5=[];N10=[]
    for iid,uid,T,cand,labs in dev_impr:
        hr,hm=hist_tensor(uid)
        ht=torch.tensor(text_idx[np.array(hr)]).unsqueeze(0).to(device)
        hmask=torch.tensor(hm,dtype=torch.bool).unsqueeze(0).to(device)
        ct=torch.tensor(text_idx[np.array([art_row.get(c,0) for c in cand])]).unsqueeze(0).to(device)
        um,hv=uvec(uid); m=len(cand)
        F=torch.tensor(np.array([[feat_vec(uid,T,iid,c,i,m,um,hv) for i,c in enumerate(cand)]]),dtype=torch.float32).to(device)
        s=model(ht,hmask,ct,F).squeeze(0).cpu().numpy()
        if cache is not None:
            for c,sc in zip(cand,s): cache[(iid,c)]=float(sc)
        a=_auc(s,labs)
        if a is not None: A.append(a)
        M.append(_mrr(s,labs));N5.append(_ndcg(s,labs,5));N10.append(_ndcg(s,labs,10))
    return dict(AUC=np.mean(A),MRR=np.mean(M),nDCG5=np.mean(N5),nDCG10=np.mean(N10))


In [ ]:
# ---- train hybrid: lr=1e-4 + grad clip + divergence guard + per-epoch push ----
from torch.utils.data import DataLoader
opt=torch.optim.Adam(model.parameters(),lr=LR)
loader=DataLoader(train_samples,batch_size=BATCH,shuffle=True,collate_fn=collate,num_workers=2)
def save(tag):
    p=f"/kaggle/working/mind_nrms_hybrid_{tag}.pt"; torch.save({"model":model.state_dict(),"vocab":vocab},p); push(p,f"mind_nrms_hybrid_{tag}.pt")
best=-1;best_ep=-1;patience=0
for ep in range(EPOCHS):
    model.train(); t0=time.time(); tot=0; nb=0; run=[]
    for ht,hm,ct,F,y in loader:
        ht,hm,ct,F,y=ht.to(device),hm.to(device),ct.to(device),F.to(device),y.to(device)
        loss=Fnn.cross_entropy(model(ht,hm,ct,F),y)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP); opt.step()
        tot+=loss.item(); nb+=1; run.append(loss.item())
        if nb==200:
            dt_=time.time()-t0; print(f"  [ep{ep}] 200 steps {dt_:.1f}s -> ~{dt_/200*len(loader)/60:.1f} min/epoch")
        if nb%2000==0:
            recent=np.mean(run[-2000:]); start=np.mean(run[:2000])
            print(f"  ep{ep} step {nb}/{len(loader)} loss {tot/nb:.4f} (recent2k {recent:.4f})")
            if nb>=10000 and recent>start+0.02:
                print("  DIVERGENCE GUARD -> stop epoch"); break
    m=eval_h()
    print(f"epoch {ep}: loss {tot/max(1,nb):.4f} | dev AUC {m['AUC']:.4f} MRR {m['MRR']:.4f} nDCG@10 {m['nDCG10']:.4f} ({(time.time()-t0)/60:.1f} min)")
    save(f"ep{ep}")
    if m["AUC"]>best: best=m["AUC"]; best_ep=ep; save("best")
    else:
        patience+=1
        if patience>EARLY_STOP_PATIENCE: print(f"early stop (best {best:.4f} @ ep{best_ep})"); break
print(f"HYBRID NRMS best dev AUC {best:.4f} @ epoch {best_ep}")


In [ ]:
# ---- load best, final eval + cache ----
ck=torch.load(dl("mind_nrms_hybrid_best.pt"), map_location=device)
model.load_state_dict(ck["model"])
cache={}; m=eval_h(cache=cache)
print("=== HYBRID NRMS (text + features) best dev ===", {k:round(v,4) for k,v in m.items()})
pickle.dump(cache, open("/kaggle/working/mind_nrms_hybrid_devscores.pkl","wb"))
push("/kaggle/working/mind_nrms_hybrid_devscores.pkl","mind_nrms_hybrid_devscores.pkl")
print("\nCompare: NRMS title+abstract 0.6589 | MLP 0.6863 | GBDT 0.7099/0.7110 | HYBRID (this)")
print("If hybrid still < GBDT: neural fails even given the same behavioural signals -> trees win on tabular.")
